# Component 4: Building the Conditioning Vector

The diffusion model's UNet does not operate in isolation -- it is **conditioned** on the current observation so that the denoised actions are appropriate for the current scene. This conditioning vector is assembled from three sources:

1. **Visual features** (128d): The image is passed through ResNet18 + SpatialSoftmax, producing 32 keypoints x 2 coordinates = 64d per image. With 2 observation timesteps, this gives 2 x 64 = 128d.

2. **Robot state** (4d): The proprioceptive state (e.g., end-effector position) from 2 timesteps, each with 2 dimensions = 4d.

3. **Timestep embedding** (128d): The sinusoidal embedding of the diffusion timestep $t$, passed through an MLP.

These are concatenated into a single vector: **128 + 4 + 128 = 260d**, which is then injected into every layer of the UNet via FiLM conditioning (Feature-wise Linear Modulation).

In this notebook, we will build the vision encoder, state encoder, and timestep encoder from scratch, then trace the entire assembly process step by step using synthetic PushT-like observations.

In [ ]:
!pip install -q torch torchvision matplotlib numpy 2>&1 | tail -3

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import torch.nn as nn
import torchvision.models as models

# ============================================================
# Build the vision encoder (ResNet18 + SpatialSoftmax) from scratch
# ============================================================

class SpatialSoftmax(nn.Module):
    """Converts a feature map into (x, y) keypoint coordinates.
    
    For each channel, applies spatial softmax to get a probability distribution
    over spatial locations, then computes the expected (x, y) coordinates.
    
    Input:  (B, C, H, W) feature map
    Output: (B, C*2) keypoint coordinates (x, y per channel)
    """
    def __init__(self, num_keypoints=32):
        super().__init__()
        self.num_keypoints = num_keypoints
        self._coord_cache = {}

    def forward(self, x):
        B, C, H, W = x.shape
        # Create coordinate grids (cached for efficiency)
        key = (H, W, x.device)
        if key not in self._coord_cache:
            ys = torch.linspace(-1, 1, H, device=x.device)
            xs = torch.linspace(-1, 1, W, device=x.device)
            grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
            self._coord_cache[key] = (grid_x, grid_y)
        grid_x, grid_y = self._coord_cache[key]
        
        # Spatial softmax: flatten spatial dims and apply softmax
        flat = x.view(B, C, -1)  # (B, C, H*W)
        attention = torch.softmax(flat, dim=-1)  # (B, C, H*W)
        
        # Expected coordinates
        expected_x = (attention * grid_x.flatten()).sum(dim=-1)  # (B, C)
        expected_y = (attention * grid_y.flatten()).sum(dim=-1)  # (B, C)
        
        return torch.cat([expected_x, expected_y], dim=-1)  # (B, C*2)


class RGBEncoder(nn.Module):
    """ResNet18 backbone + SpatialSoftmax pooling.
    
    Takes a 96x96 RGB image and produces a 64d feature vector:
    32 keypoints x 2 coordinates = 64 dimensions.
    """
    def __init__(self, num_keypoints=32):
        super().__init__()
        # Use pretrained ResNet18, remove avgpool and fc
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4,
        )
        # Project 512 channels down to num_keypoints channels
        self.channel_proj = nn.Conv2d(512, num_keypoints, 1)
        self.pool = SpatialSoftmax(num_keypoints)
    
    def forward(self, x):
        features = self.backbone(x)         # (B, 512, 3, 3)
        features = self.channel_proj(features)  # (B, 32, 3, 3)
        return self.pool(features)          # (B, 64)


class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for diffusion timesteps."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=x.device, dtype=torch.float32) * -emb)
        emb = x.unsqueeze(-1).float() * emb.unsqueeze(0)
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class DiffusionStepEncoder(nn.Module):
    """Timestep encoder: sinusoidal -> MLP -> embedding."""
    def __init__(self, dim=128):
        super().__init__()
        self.net = nn.Sequential(
            SinusoidalPosEmb(dim),
            nn.Linear(dim, dim * 4),
            nn.Mish(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x):
        return self.net(x)


# Instantiate components
rgb_encoder = RGBEncoder(num_keypoints=32).to(device)
rgb_encoder.eval()
timestep_encoder = DiffusionStepEncoder(dim=128).to(device)

# ============================================================
# Create synthetic observation data (simulates PushT dataset)
# ============================================================
torch.manual_seed(42)

def make_synthetic_scene(seed=0):
    """Create a synthetic 96x96 scene simulating a PushT workspace.
    Returns 2 observation images and 2 state vectors (2 timesteps).
    """
    rng = torch.Generator().manual_seed(seed)
    images = torch.zeros(2, 3, 96, 96)
    states = torch.zeros(2, 2)
    
    for t in range(2):
        # Random colored blocks on a gray background
        img = torch.ones(3, 96, 96) * 0.3
        # T-shaped object (blue)
        x_off = int(torch.randint(20, 60, (1,), generator=rng).item())
        y_off = int(torch.randint(20, 60, (1,), generator=rng).item())
        img[2, y_off:y_off+8, x_off:x_off+20] = 0.9   # horizontal bar
        img[2, y_off+8:y_off+25, x_off+7:x_off+13] = 0.9  # vertical bar
        # End-effector (red dot)
        ex = int(torch.randint(10, 85, (1,), generator=rng).item())
        ey = int(torch.randint(10, 85, (1,), generator=rng).item())
        img[0, ey-3:ey+3, ex-3:ex+3] = 0.95
        # Add noise
        img += torch.randn_like(img) * 0.03
        images[t] = img.clamp(0, 1)
        states[t] = torch.tensor([ex / 96.0, ey / 96.0])
    
    return images, states


# Create 6 different synthetic scenes for demonstrations
synthetic_scenes = {}
for seed in [0, 42, 100, 500, 1000, 5000]:
    synthetic_scenes[seed] = make_synthetic_scene(seed)

# ImageNet normalization (same as diffusion policy)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

print("Components built from scratch:")
print(f"  - RGB Encoder: ResNet18 + SpatialSoftmax -> 64d per image")
print(f"  - Timestep Encoder: Sinusoidal + MLP -> 128d")
print(f"  - Synthetic scenes: {len(synthetic_scenes)} scenes created")
print(f"\nConditioning vector = visual(128d) + state(4d) + timestep(128d) = 260d")

## Step-by-Step Assembly

We will load a synthetic sample and trace every encoding step, printing shapes and values at each stage.

In [ ]:
# Load a synthetic sample
images, state = synthetic_scenes[100]
images = images.to(device)   # (2, 3, 96, 96)
state = state.to(device)     # (2, 2)

print("=" * 60)
print("RAW INPUTS")
print("=" * 60)
print(f"Images shape: {images.shape}")
print(f"  -> 2 timesteps x 3 channels (RGB) x 96 x 96 pixels")
print(f"  -> Image pixel range: [{images.min().item():.3f}, {images.max().item():.3f}]")
print(f"\nState shape:  {state.shape}")
print(f"  -> 2 timesteps x 2 dims (x, y end-effector position)")
print(f"  -> State values: {state.cpu().numpy().round(4)}")

# Show the two observation images
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i in range(2):
    img = images[i].cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    axes[i].imshow(img)
    axes[i].set_title(f"Observation t={['t-1', 't'][i]}")
    axes[i].axis("off")
plt.suptitle("Input Observation Images (Synthetic PushT Scene)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 1: Image -> ResNet18 -> SpatialSoftmax -> 64d per image
# ============================================================
print("=" * 60)
print("STEP 1: Visual Encoding (Image -> 64d feature vector)")
print("=" * 60)

# Normalize images for ResNet
images_norm = (images - MEAN.to(device)) / STD.to(device)

# Process each image separately to show the pipeline
visual_features_list = []
with torch.no_grad():
    for t_idx in range(2):
        img = images_norm[t_idx:t_idx+1]  # (1, 3, 96, 96)
        
        # ResNet18 backbone
        backbone_out = rgb_encoder.backbone(img)  # (1, 512, H, W)
        print(f"  Image t={t_idx}: {img.shape} -> ResNet18 -> {backbone_out.shape}")
        
        # Channel projection + SpatialSoftmax pooling
        projected = rgb_encoder.channel_proj(backbone_out)  # (1, 32, H, W)
        pooled = rgb_encoder.pool(projected)  # (1, 64)
        print(f"  {backbone_out.shape} -> proj -> {projected.shape} -> SpatialSoftmax -> {pooled.shape}")
        
        visual_features_list.append(pooled)

# Concatenate visual features from both timesteps
visual_features = torch.cat(visual_features_list, dim=-1)  # (1, 128)
print(f"\n  Concatenated visual features: {visual_features.shape}")
print(f"  = 2 images x 32 keypoints x 2 coords = {2*32*2}d")

# ============================================================
# STEP 2: State features (already low-dimensional)
# ============================================================
print(f"\n{'=' * 60}")
print("STEP 2: State Features (already compact)")
print("=" * 60)

state_flat = state.flatten().unsqueeze(0)  # (1, 4)
print(f"  State: {state.shape} -> flatten -> {state_flat.shape}")
print(f"  = 2 timesteps x 2 dims (x, y) = {2*2}d")
print(f"  Values: {state_flat[0].cpu().numpy().round(4)}")

# ============================================================
# STEP 3: Timestep embedding
# ============================================================
print(f"\n{'=' * 60}")
print("STEP 3: Timestep Embedding")
print("=" * 60)

# Pick a diffusion timestep
diffusion_t = torch.tensor([50], device=device)  # midway through denoising

with torch.no_grad():
    timestep_emb = timestep_encoder(diffusion_t)  # (1, 128)

print(f"  Diffusion timestep: t={diffusion_t.item()}")
print(f"  Timestep embedding shape: {timestep_emb.shape}")
print(f"  = sinusoidal(128d) -> MLP -> {timestep_emb.shape[-1]}d")

# ============================================================
# STEP 4: Assemble the full conditioning vector
# ============================================================
print(f"\n{'=' * 60}")
print("STEP 4: Assemble Conditioning Vector")
print("=" * 60)

conditioning = torch.cat([visual_features, state_flat, timestep_emb], dim=-1)
print(f"  Visual features: {visual_features.shape[-1]}d")
print(f"  + State features: {state_flat.shape[-1]}d")
print(f"  + Timestep embedding: {timestep_emb.shape[-1]}d")
print(f"  " + "-" * 40)
print(f"  = Conditioning vector: {conditioning.shape[-1]}d")
print(f"\n  This {conditioning.shape[-1]}d vector is injected into every")
print(f"  layer of the UNet via FiLM conditioning.")

In [ ]:
# Visualize the assembled conditioning vector as colored segments
vis_dim = visual_features.shape[-1]
state_dim = state_flat.shape[-1]
time_dim = timestep_emb.shape[-1]
total_dim = vis_dim + state_dim + time_dim

cond_np = conditioning[0].cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={"height_ratios": [1, 3]})

# Top: Segment diagram
ax = axes[0]
segments = [
    ("Visual (img t-1)\n64d", 0, vis_dim // 2, "#D97757"),
    ("Visual (img t)\n64d", vis_dim // 2, vis_dim, "#C4956A"),
    ("State\n4d", vis_dim, vis_dim + state_dim, "#7DA488"),
    ("Timestep\n" + f"{time_dim}d", vis_dim + state_dim, total_dim, "#5B8CB8"),
]

for label, start, end, color in segments:
    ax.barh(0, end - start, left=start, height=0.6, color=color,
            edgecolor="white", linewidth=2)
    mid = (start + end) / 2
    ax.text(mid, 0, label, ha="center", va="center", fontsize=10,
            fontweight="bold", color="white")

ax.set_xlim(0, total_dim)
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel(f"Dimension (total = {total_dim}d)", fontsize=12)
ax.set_yticks([])
ax.set_title("Conditioning Vector Structure", fontsize=14, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

# Bottom: Actual values
ax2 = axes[1]
x_range = np.arange(total_dim)

# Color each segment
colors_array = np.empty(total_dim, dtype=object)
for _, start, end, color in segments:
    colors_array[start:end] = color

ax2.bar(x_range, cond_np, color=colors_array, width=1.0, edgecolor="none")
ax2.set_xlabel("Dimension Index", fontsize=12)
ax2.set_ylabel("Value", fontsize=12)
ax2.set_title("Actual Values in the Conditioning Vector", fontsize=13, fontweight="bold")
ax2.axhline(y=0, color="gray", linestyle="-", alpha=0.3)

# Add vertical lines at segment boundaries
for _, start, end, _ in segments:
    if start > 0:
        ax2.axvline(x=start, color="white", linestyle="-", linewidth=2, alpha=0.8)

ax2.set_xlim(0, total_dim)
ax2.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\nValue statistics per segment:")
for label, start, end, _ in segments:
    seg = cond_np[start:end]
    lbl = label.split('\n')[0]
    print(f"  {lbl:20s}: mean={seg.mean():.4f}, std={seg.std():.4f}, "
          f"range=[{seg.min():.4f}, {seg.max():.4f}]")

In [ ]:
# Compare conditioning vectors from different scenes
print("=" * 60)
print("Comparing Conditioning Vectors from Different Scenes")
print("=" * 60)

scene_seeds = [0, 42, 100, 1000, 5000]
all_cond_vectors = []

fig_imgs, axes_imgs = plt.subplots(1, len(scene_seeds), figsize=(15, 3))

with torch.no_grad():
    for plot_idx, seed in enumerate(scene_seeds):
        imgs, st = synthetic_scenes[seed]
        imgs = imgs.to(device)
        st = st.to(device)
        
        # Normalize images
        imgs_norm = (imgs - MEAN.to(device)) / STD.to(device)
        
        # Encode images
        vis_feats = []
        for t_idx in range(2):
            pooled = rgb_encoder(imgs_norm[t_idx:t_idx+1])
            vis_feats.append(pooled)
        vis = torch.cat(vis_feats, dim=-1)  # (1, 128)
        
        # State
        st_flat = st.flatten().unsqueeze(0)  # (1, 4)
        
        # Timestep (use same t=50 for fair comparison)
        t_emb = timestep_emb  # reuse from above
        
        cond = torch.cat([vis, st_flat, t_emb], dim=-1)  # (1, 260)
        all_cond_vectors.append(cond[0].cpu())
        
        # Show image
        img_show = imgs[1].cpu().permute(1, 2, 0).numpy()
        img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min() + 1e-8)
        axes_imgs[plot_idx].imshow(img_show)
        axes_imgs[plot_idx].set_title(f"Scene {seed}", fontsize=10)
        axes_imgs[plot_idx].axis("off")

plt.suptitle("Scenes Being Compared", fontweight="bold")
plt.tight_layout()
plt.show()

# Cosine similarity between conditioning vectors
cond_stack = torch.stack(all_cond_vectors)  # (5, 260)
cond_norm = cond_stack / cond_stack.norm(dim=1, keepdim=True)
sim_matrix = cond_norm @ cond_norm.T

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(sim_matrix.numpy(), cmap="viridis", vmin=-0.2, vmax=1.0)
ax.set_xticks(range(len(scene_seeds)))
ax.set_yticks(range(len(scene_seeds)))
ax.set_xticklabels([f"s{i}" for i in scene_seeds])
ax.set_yticklabels([f"s{i}" for i in scene_seeds])
ax.set_title("Cosine Similarity Between\nConditioning Vectors", fontweight="bold")

# Add text annotations
for i in range(len(scene_seeds)):
    for j in range(len(scene_seeds)):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center",
                fontsize=9, color="white" if sim_matrix[i,j] < 0.5 else "black")

plt.colorbar(im, label="Cosine Similarity")
plt.tight_layout()
plt.show()

print("\nDifferent scenes produce different conditioning vectors.")
print("The visual features capture WHAT the scene looks like,")
print("the state captures WHERE the robot is, and the timestep")
print("tells the UNet HOW noisy the current actions are.")

In [ ]:
# Show what each segment "looks like" individually
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

segments_data = [
    ("Visual (img t-1)", cond_np[:vis_dim//2], "#D97757"),
    ("Visual (img t)", cond_np[vis_dim//2:vis_dim], "#C4956A"),
    ("Robot State", cond_np[vis_dim:vis_dim+state_dim], "#7DA488"),
    ("Timestep Emb", cond_np[vis_dim+state_dim:], "#5B8CB8"),
]

for ax, (label, data, color) in zip(axes, segments_data):
    ax.bar(range(len(data)), data, color=color, alpha=0.8, width=1.0)
    ax.set_title(f"{label}\n({len(data)}d)", fontweight="bold", fontsize=11)
    ax.set_xlabel("Dim index")
    ax.set_ylabel("Value")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    ax.grid(axis="y", alpha=0.2)
    
    # Show interpretation
    if "State" in label:
        for i, val in enumerate(data):
            dim_names = ["x(t-1)", "y(t-1)", "x(t)", "y(t)"]
            if i < len(dim_names):
                ax.text(i, val + 0.02 * np.sign(val), dim_names[i],
                        ha="center", fontsize=8, fontweight="bold")

plt.suptitle("Individual Segments of the Conditioning Vector",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey insight: The visual features are (x,y) keypoint coordinates,")
print("so they oscillate around 0 (normalized to [-1, 1]).")
print("The state features are raw robot positions.")
print("The timestep embedding has a characteristic sinusoidal + MLP pattern.")

## TODO Exercise

**If you had 3 cameras instead of 1, what would the conditioning vector dimension be?**

1. Calculate the new dimension:
   - Visual features: 3 cameras x 2 timesteps x 32 keypoints x 2 coords = ???d
   - State features: still 4d (robot state does not depend on cameras)
   - Timestep embedding: still 128d
   - Total = ???d

2. Would this cause any issues?
   - The UNet's FiLM conditioning layers expect a fixed input dimension
   - You would need to update `cfg.input_features` to include all 3 camera keys
   - The model would automatically adjust its conditioning MLP

3. Experiment:
```python
# TODO: Create a new config with 3 cameras
# input_features_3cam = {
#     "observation.image_left": ...,
#     "observation.image_right": ...,
#     "observation.image_wrist": ...,
#     "observation.state": ...,
# }
# cfg_3cam = DiffusionConfig(input_features=input_features_3cam, ...)
# What changes in the model architecture? Print the new model and compare.
```

4. Real-world context: The ALOHA bimanual manipulation setup uses 4 cameras. How large would the conditioning vector be?